In [ ]:
import sys
sys.path.append('/var/www/python/Prod/nighthawk/')

import pandas as pd
from pathlib import Path
from nighthawk.data import Constraint

CSV_PATH  = Path('/var/www/python/Qingcheng/QCTest/Manual_bidding/update_sheet/Daily Bidding - daily_constraint.csv')
SOURCE    = 'csv'          # 'sheet' reads/writes the Google Sheet, 'csv' the local file
MARKET    = 'SPP'
THRESHOLD = -900
META_COLS = ['location', 'physical_condition', 'outage_name',
             'comment on this constraint', 'start_date', 'end_date',
             'wind', 'reserve_zone']
COL_ORDER = ['bid_date', 'monitored', 'DA_mvalue', 'RT_mvalue',
             'location', 'physical_condition', 'outage_name',
              'start_date', 'end_date',
             'wind', 'load', 'reserve_zone', "today's wind", 'opportunity']


def _read_source() -> pd.DataFrame:
    """Load the current bidding table from the Google Sheet or the local CSV."""
    if SOURCE == 'sheet':
        ws = get_worksheet()
        df = get_as_dataframe(ws, evaluate_formulas=True, dtype=str)
        df = df.dropna(axis=1, how='all').dropna(axis=0, how='all')   # trailing blank rows/cols
        print(f'read {len(df)} rows from sheet tab "{ws.title}"')
        return df
    return pd.read_csv(CSV_PATH)


def _write_target(result: pd.DataFrame) -> None:
    """
    Write the table back to wherever it was read from, after a local backup.

    The sheet write clears the tab and rewrites it, so anything outside COL_ORDER is
    lost — the same semantics the CSV path always had, but on a shared document, so
    the timestamped backup is written first and unconditionally.
    """
    out = result.rename(columns={'monitored': 'constraints '})

    stamp  = pd.Timestamp.now(tz='US/Central').strftime('%Y%m%d_%H%M%S')
    backup = CSV_PATH.with_name(f'{CSV_PATH.stem}.backup_{stamp}.csv')
    out.to_csv(backup, index=False)
    print(f'backup  → {backup}')

    if SOURCE == 'sheet':
        ws = get_worksheet()
        ws.clear()
        set_with_dataframe(ws, out.fillna(''), resize=True)
        print(f'wrote {len(out)} rows → sheet tab "{ws.title}"')
    else:
        out.to_csv(CSV_PATH, index=False)
        print(f'Saved → {CSV_PATH}')


def _fetch_mvalues(dt_str: str) -> pd.DataFrame:
    opex  = MARKET
    mv_rt = Constraint(oops_constraint_num_df=None, market=opex).get_mvalues(
        start_dt=dt_str, end_dt=dt_str, type='RT', granularity='daily')
    mv_da = Constraint(oops_constraint_num_df=None, market=opex).get_mvalues(
        start_dt=dt_str, end_dt=dt_str, type='DA', granularity='daily')

    all_cons = pd.DataFrame({'oops_constraint_num':
        pd.concat([mv_rt['oops_constraint_num'], mv_da['oops_constraint_num']]).unique()})

    det_rt = Constraint(oops_constraint_num_df=all_cons, market=opex).get_constraint_details(da_or_rt='RT')
    det_da = Constraint(oops_constraint_num_df=all_cons, market=opex).get_constraint_details(da_or_rt='DA')
    details = (pd.concat([det_rt, det_da])
               .drop_duplicates('oops_constraint_num')
               [['oops_constraint_num', 'monitored_clean']]
               .rename(columns={'monitored_clean': 'monitored'}))

    da_sum = (mv_da.merge(details, on='oops_constraint_num', how='left')
              .groupby('monitored')['mvalue'].sum().rename('DA_mvalue').reset_index())
    rt_sum = (mv_rt.merge(details, on='oops_constraint_num', how='left')
              .groupby('monitored')['mvalue'].sum().rename('RT_mvalue').reset_index())

    merged = pd.merge(da_sum, rt_sum, on='monitored', how='outer').fillna(0)
    if merged.empty:
        return pd.DataFrame(columns=['monitored', 'DA_mvalue', 'RT_mvalue'])
    merged['monitored'] = merged['monitored'].str.strip()
    # coerce to numeric (column can be object dtype if a side was empty) before rounding
    merged['DA_mvalue'] = pd.to_numeric(merged['DA_mvalue'], errors='coerce').fillna(0).round(0).astype(int)
    merged['RT_mvalue'] = pd.to_numeric(merged['RT_mvalue'], errors='coerce').fillna(0).round(0).astype(int)
    return merged


def _lookup_metadata(monitored_name: str, df: pd.DataFrame) -> dict:
    prior = df[df['monitored'] == monitored_name]
    if prior.empty:
        return {c: '' for c in META_COLS}
    return {c: prior.sort_values('bid_date').iloc[-1].get(c, '') for c in META_COLS}


def _opportunity(monitored_name: str, df: pd.DataFrame, before_dt) -> str:
    prior = df[(df['monitored'] == monitored_name) & (df['bid_date'] < before_dt)]
    if prior.empty:
        return 'new'
    return pd.Timestamp(prior['bid_date'].max()).strftime('%-m/%-d/%Y')


def update_constraints(start_dt: str, end_dt: str, save: bool = True) -> pd.DataFrame:
    df = _read_source()
    df.columns = df.columns.str.strip()
    df = df.rename(columns={'constraints': 'monitored', 'constraints ': 'monitored'})
    df['monitored'] = df['monitored'].str.strip()
    # coerce bad/missing-year bid_dates (e.g. "6/15") to NaT, then drop those rows
    df['bid_date']  = pd.to_datetime(df['bid_date'], format='mixed', errors='coerce')
    df = df.dropna(subset=['bid_date']).reset_index(drop=True)

    date_range = pd.date_range(start=start_dt, end=end_dt, freq='D')
    new_rows   = []

    for dt in date_range:
        dt_str = dt.strftime('%Y-%m-%d')
        print(f"\nFetching {dt_str}...")

        fetched = _fetch_mvalues(dt_str)
        fetched = fetched[
            (fetched['DA_mvalue'] <= THRESHOLD) | (fetched['RT_mvalue'] <= THRESHOLD)
        ].reset_index(drop=True)

        if fetched.empty:
            print(f"  no constraints below threshold")
            continue
        print(f"  {len(fetched)} constraint(s) found")

        existing_mask = df['bid_date'] == dt

        if existing_mask.any():
            for _, frow in fetched.iterrows():
                row_mask = existing_mask & (df['monitored'] == frow['monitored'])
                opp = _opportunity(frow['monitored'], df, before_dt=dt)
                if row_mask.any():
                    df.loc[row_mask, 'DA_mvalue']   = frow['DA_mvalue']
                    df.loc[row_mask, 'RT_mvalue']   = frow['RT_mvalue']
                    df.loc[row_mask, 'opportunity'] = opp
                    print(f"    updated  : {frow['monitored']} (opportunity={opp})")
                else:
                    meta = _lookup_metadata(frow['monitored'], df)
                    new_rows.append({'bid_date': dt, 'monitored': frow['monitored'],
                                     'DA_mvalue': frow['DA_mvalue'], 'RT_mvalue': frow['RT_mvalue'],
                                     **meta, "today's wind": '', 'opportunity': opp})
                    print(f"    appended : {frow['monitored']} (new for this date, opportunity={opp})")
        else:
            for _, frow in fetched.iterrows():
                meta = _lookup_metadata(frow['monitored'], df)
                opp  = _opportunity(frow['monitored'], df, before_dt=dt)
                new_rows.append({'bid_date': dt, 'monitored': frow['monitored'],
                                 'DA_mvalue': frow['DA_mvalue'], 'RT_mvalue': frow['RT_mvalue'],
                                 **meta, "today's wind": '', 'opportunity': opp})
                print(f"    appended : {frow['monitored']} (opportunity={opp})")

    result = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)
    result['bid_date'] = pd.to_datetime(result['bid_date'], format='mixed', errors='coerce')
    result = result.dropna(subset=['bid_date']).reset_index(drop=True)

    # Sort: by date first, then within each date by abs(RT - DA) descending
    result['_rank'] = (
        pd.to_numeric(result['RT_mvalue'], errors='coerce').fillna(0) -
        pd.to_numeric(result['DA_mvalue'], errors='coerce').fillna(0)
    ).abs()
    result = (result
              .sort_values(['bid_date', '_rank'], ascending=[True, False])
              .drop(columns='_rank')
              .reset_index(drop=True))

    result['bid_date'] = result['bid_date'].dt.strftime('%-m/%-d/%Y')
    result = result[COL_ORDER]

    print(f"\n{'='*60}")
    print(f"Total rows: {len(result)}  |  New rows added: {len(new_rows)}")
    display(result.tail(len(new_rows) + 3))

    if save:
        _write_target(result)

    return result





In [7]:
# today    = pd.Timestamp.now(tz='US/Central').normalize().tz_localize(None)
today = pd.Timestamp('2026-07-27').normalize().tz_localize(None)
start_dt = (today - pd.Timedelta(days=1)).strftime('%Y-%m-%d')
end_dt   = (today + pd.Timedelta(days=1)).strftime('%Y-%m-%d')

result = update_constraints(start_dt, end_dt, save=True)


Fetching 2026-07-26...
  33 constraint(s) found
    updated  : ln29eveng3-29ga (opportunity=7/25/2026)
    updated  : lnanta_tp-anita (opportunity=7/2/2026)
    updated  : lnbuln-kel (opportunity=7/25/2026)
    updated  : lncommtap4-bois_tap (opportunity=7/10/2026)
    updated  : lnconcor3-clifto1 (opportunity=new)
    updated  : lncraig-lenexa (opportunity=7/14/2026)
    updated  : lnfirstcrk-roanrdge (opportunity=7/21/2026)
    updated  : lnhiltop_e-st_joe_e (opportunity=new)
    updated  : lniatan-easttown (opportunity=new)
    updated  : lnindep2-morefl (opportunity=7/19/2026)
    updated  : lnjamestn-valleyc (opportunity=7/15/2026)
    updated  : lnlacygne-stilwell (opportunity=7/9/2026)
    updated  : lnmaud_tap-earlsbrh (opportunity=7/8/2026)
    updated  : lnmonett-aur1241 (opportunity=7/16/2026)
    updated  : lnnebrcty-sub3456 (opportunity=7/19/2026)
    updated  : lnnorfork-soulan (opportunity=7/25/2026)
    updated  : lnommarlo4-omdune-4 (opportunity=new)
    updated  : ln

,bid_date,monitored,DA_mvalue,RT_mvalue,location,physical_condition,outage_name,start_date,end_date,wind,load,reserve_zone,today's wind,opportunity
907,7/27/2026,ln29eveng3-29ga,-4470.0,-4085.0,West Kansas City,"outage drive, high load, all wind",Auburn WR - Jeffrey Energy Center 230 kV,2026-06-04 23:33,2026-06-05 16:00,mid,NaN,4,NaN,7/26/2026
908,7/27/2026,lnjeff-hoyt,-994.0,-846.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,new
909,7/27/2026,lnstilwell-redel5,-1881.0,-1773.0,South Kansas,"binds in very high wind DA 50%, RT 20%, no obv...",NaN,NaN,NaN,high,NaN,4,NaN,7/26/2026
910,7/28/2026,lnst_joe-avectytp,-3520.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,7/27/2026
911,7/28/2026,lnnebrcty-sub3456,-3199.0,0.0,"S-N, Omaha","984kv, low wind, 35% RT low wind, 25% high load",NaN,NaN,NaN,low,NaN,1,,7/27/2026
912,7/28/2026,lnt_brmghm-lbrtyst5,-2708.0,0.0,Kansas,"200kv, S-N, load driven",NaN,NaN,NaN,mid,NaN,4,,7/27/2026
913,7/28/2026,ln29eveng3-29ga,-2315.0,0.0,West Kansas City,"outage drive, high load, all wind",Auburn WR - Jeffrey Energy Center 230 kV,2026-06-04 23:33,2026-06-05 16:00,mid,NaN,4,,7/27/2026
914,7/28/2026,lnrussett-sbrown,-1509.0,0.0,South OKGE,"binds high wind, no obvious outage nearby",NaN,NaN,NaN,high,NaN,"3,4",,7/27/2026
915,7/28/2026,lnmarshal3-knob,-1480.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,7/27/2026
916,7/28/2026,lnsnakeck-aliance,-1064.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,7/27/2026


Saved → /var/www/python/Qingcheng/QCTest/Manual_bidding/update_sheet/Daily Bidding - daily_constraint.csv
